In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Konfiguration
data_path = 'mnist_train.csv'
num_classes = 10
batch_size = 128
learning_rate = 0.01
num_epochs = 3
train_split = 0.8

data = pd.read_csv(data_path)

labels = data.iloc[:, 0].values
pixels = data.iloc[:, 1:].values

images = pixels.reshape(-1, 1, 28, 28).astype(np.float32) / 255.0
labels_tensor = labels.astype(np.int64)

# Zu PyTorch Tensoren konvertieren
X = torch.FloatTensor(images)
y = torch.LongTensor(labels_tensor)

# Batch-Größe festlegen
dataset = TensorDataset(X, y)

# Train/Test Split
train_size = int(train_split * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset, 
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# DataLoader erstellen
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


# Input: 28x28
# Nach Conv1 (kernel=5, padding=2): (28 - 5 + 2*2) / 1 + 1 = 28 -> bleibt 28x28
# Nach MaxPool1 (kernel=2, stride=2): 28 / 2 = 14 -> wird 14x14
# Nach Conv2 (kernel=5, padding=2): (14 - 5 + 2*2) / 1 + 1 = 14 -> bleibt 14x14
# Nach MaxPool2 (kernel=2, stride=2): 14 / 2 = 7 -> wird 7x7
# End-Dimension pro Bild: 64 Filter * 7 * 7 Pixel
# nn.Linear(64 * 7 * 7, 128),

model = nn.Sequential(
    # Erster Convolutional Block
    nn.Conv2d(1, 32, kernel_size=5, padding=2),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),
    
    # Zweiter Convolutional Block
    nn.Conv2d(32, 64, kernel_size=5, padding=2),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),
    
    # Fully Connected Layers
    nn.Flatten(),
    nn.Linear(64 * 7 * 7, 128),
    nn.BatchNorm1d(128),  # Batch Normalization
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, num_classes)
)

# Loss und Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, target)
        
        # Backward pass
        loss.backward()
        optimizer.step()

model.eval()
test_loss = 0.0
correct = 0
total = 0
all_predictions = []
all_targets = []

with torch.no_grad():
    for batch_idx, (data, target) in enumerate(test_loader):
        
        outputs = model(data)
        test_loss += criterion(outputs, target).item()
        
        _, predicted = torch.max(outputs, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
        
        all_predictions.extend(predicted.cpu().numpy())
        all_targets.extend(target.cpu().numpy())

# Finale Ergebnisse
test_loss = test_loss / len(test_loader)
test_accuracy = 100. * correct / total

print(f'\nTEST-ERGEBNISSE:')
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.2f}% ({correct}/{total})')


TEST-ERGEBNISSE:
Test Loss: 0.0474
Test Accuracy: 98.57% (11828/12000)
